# Developer Lifecycle Analysis - Clean Pipeline

## Project Context
This notebook analyzes NVIDIA developer engagement to identify how developers progress through the platform and where they drop off. The analysis classifies developers into three populations:

1. **Tourists** - one-time visitors who never returned
2. **Free Email Users** - returners who only download free assets
3. **Real Users** - genuinely engaged developers analyzed for stage and dormancy

## Architectural Note
This pipeline replaces the original `activity_ontology_v1` table which had a fundamental issue: it was a 69-million-row materialized table where the same activity received different classifications across different rows. For example, "forum contributions" appeared as Champion in some rows and Build in others, depending on persona detection logic that was conflated with stage classification.

This caused two problems:
- Counts were inconsistent (the same activity could fall into multiple buckets)
- Joins were extremely slow (anything joined to the 69M table took 20+ minutes)

The fix follows standard dimensional modeling: separate the small dictionary (22 rows, one per activity) from the large fact table (activity events). Each activity gets exactly one classification. Joins are instant. Counts are consistent.

## Pipeline Steps
1. Setup and verify sample tables
2. Build clean activity dictionary (22 rows)
3. Enrich sample with deterministic classifications
4. Apply activation gate (filter Tourists and Free Email Users)
5. Assign max stage reached for Real Users
6. Apply dormancy classification
7. Combine into final status
8. Export results

In [1]:
import duckdb
import os
import pandas as pd

con = duckdb.connect("developer_project.duckdb")
con.execute("PRAGMA memory_limit='6GB'")
con.execute("PRAGMA threads=4")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

print("Connected to developer_project.duckdb")
print("\nVerifying sample tables exist:")
print(con.execute("""
SELECT 
    'activity_sample' AS table_name, 
    COUNT(*) AS rows,
    COUNT(DISTINCT dev_contact) AS distinct_developers
FROM activity_sample
UNION ALL
SELECT 
    'contact_sample',
    COUNT(*),
    COUNT(DISTINCT developer_id)
FROM contact_sample
""").fetchdf())

Connected to developer_project.duckdb

Verifying sample tables exist:
        table_name    rows  distinct_developers
0  activity_sample  200000               147107
1   contact_sample  147105               147105


## Step 1: Build the Activity Dictionary

This is the architectural fix. Instead of using the 69-million-row `activity_ontology_v1` table that produces inconsistent classifications, we create a small 22-row dictionary where each unique activity has exactly one journey stage and effort level.

### Classification Decisions
Each activity is classified based on NVIDIA's public documentation and scoring guideline:

- **Brev** classified as Build/High: cloud GPU instance provisioning is active development
- **NGC Downloads** classified as Build/High: containers, models, and Helm charts are deployment artifacts
- **DevZone Downloads** classified as Evaluate/Moderate by default, but split by file_type at enrichment time:
  - Installer/toolkit files → Build/High
  - Documentation/PDFs → Discover/Passive
  - Other downloads → Evaluate/Moderate
- **Product Specific Comms** classified as Discover/Passive: opt-in marketing only
- **Forum Contributions** classified as Champion/High: community advocacy
- **DLI Training** classified as Learn/Moderate: structured learning content

In [2]:
con.execute("""
CREATE OR REPLACE TABLE activity_dict_clean AS
SELECT * FROM (VALUES
    ('event registrations',    'Discover', 'Passive',  'Event'),
    ('eventy registrations',   'Discover', 'Passive',  'Event'),
    ('other events',           'Discover', 'Passive',  'Event'),
    ('product specific comms', 'Discover', 'Passive',  'Communication'),
    ('on-demand views',        'Discover', 'Passive',  'On Demand'),
    ('dev program membership', 'Discover', 'Passive',  'Membership'),
    ('webinars',               'Learn',    'Moderate', 'Event'),
    ('conference',             'Learn',    'Moderate', 'Event'),
    ('conf sessions live',     'Learn',    'Moderate', 'Event'),
    ('conf. sessions live',    'Learn',    'Moderate', 'Event'),
    ('dli training',           'Learn',    'Moderate', 'Training'),
    ('devzone downloads',      'Evaluate', 'Moderate', 'Download'),
    ('program applications',   'Evaluate', 'Moderate', 'Application'),
    ('ngc downloads',          'Build',    'High',     'Download'),
    ('hosted api',             'Build',    'High',     'Hosted API'),
    ('model api',              'Build',    'High',     'Hosted API'),
    ('hackathon',              'Build',    'High',     'Application'),
    ('hackathons',             'Build',    'High',     'Application'),
    ('brev',                   'Build',    'High',     'Cloud Workspace'),
    ('forum contributions',    'Champion', 'High',     'Community'),
    ('bugs filed',             'Champion', 'High',     'Support Feedback'),
    ('user feedback',          'Champion', 'High',     'Support Feedback'),
    ('contests',               'Champion', 'High',     'Community')
) AS t(activity, journey_signal, effort_level, modality)
""")

print("Dictionary contents:")
print(con.execute("SELECT * FROM activity_dict_clean ORDER BY journey_signal, activity").fetchdf())

Dictionary contents:
                  activity journey_signal effort_level          modality
0                     brev          Build         High   Cloud Workspace
1                hackathon          Build         High       Application
2               hackathons          Build         High       Application
3               hosted api          Build         High        Hosted API
4                model api          Build         High        Hosted API
5            ngc downloads          Build         High          Download
6               bugs filed       Champion         High  Support Feedback
7                 contests       Champion         High         Community
8      forum contributions       Champion         High         Community
9            user feedback       Champion         High  Support Feedback
10  dev program membership       Discover      Passive        Membership
11     event registrations       Discover      Passive             Event
12    eventy registrations    

In [3]:
print("Coverage check (UNMAPPED rows would need to be added to dictionary):")
print(con.execute("""
SELECT 
    LOWER(TRIM(a.activity)) AS activity,
    COUNT(*) AS rows,
    CASE WHEN d.activity IS NOT NULL THEN 'mapped' ELSE 'UNMAPPED' END AS status
FROM activity_sample a
LEFT JOIN activity_dict_clean d
  ON LOWER(TRIM(a.activity)) = d.activity
GROUP BY 1, 3
ORDER BY status DESC, rows DESC
""").fetchdf())

Coverage check (UNMAPPED rows would need to be added to dictionary):
                  activity    rows  status
0        devzone downloads  114846  mapped
1            ngc downloads   22155  mapped
2   dev program membership   19535  mapped
3                model api   18607  mapped
4             dli training    5559  mapped
5          on-demand views    4076  mapped
6      conf. sessions live    3805  mapped
7            user feedback    3582  mapped
8      forum contributions    2202  mapped
9   product specific comms    2093  mapped
10              conference    1194  mapped
11                webinars     938  mapped
12    program applications     653  mapped
13                    brev     358  mapped
14              bugs filed     333  mapped
15     event registrations      28  mapped
16            other events      28  mapped
17              hackathons       4  mapped
18                contests       4  mapped


## Step 2: Enrich the Activity Sample

We join each activity event to the dictionary to apply classifications. We also implement DevZone file_type splitting at this layer using filepath patterns.

DevZone splitting logic:
- Filenames matching `installer`, `toolkit`, `.exe`, `.deb`, `.rpm` → Build/High
- Filepaths matching `.pdf`, `docs/`, `documentation` → Discover/Passive
- Everything else → Evaluate/Moderate (the dictionary default)

This is the only place where context-dependent classification happens, and it's based on file_type which NVIDIA's own scoring guideline differentiates by.

In [4]:
con.execute("""
CREATE OR REPLACE TABLE activity_sample_enriched AS
SELECT 
    a.dev_contact AS developer_id,
    CAST(a.activity_date AS DATE) AS activity_date,
    DATE_TRUNC('week', CAST(a.activity_date AS DATE)) AS week_start,
    LOWER(TRIM(a.activity)) AS activity_clean,
    a.activity_name,
    a.filepath,
    -- DevZone file_type override for journey_signal
    CASE
        WHEN LOWER(TRIM(a.activity)) = 'devzone downloads' THEN
            CASE
                WHEN LOWER(COALESCE(a.filepath, '')) LIKE '%.exe' 
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%installer%'
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%toolkit%'
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%.deb'
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%.rpm'
                THEN 'Build'
                WHEN LOWER(COALESCE(a.filepath, '')) LIKE '%.pdf'
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%docs/%'
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%documentation%'
                THEN 'Discover'
                ELSE 'Evaluate'
            END
        ELSE d.journey_signal
    END AS journey_signal,
    -- DevZone file_type override for effort_level
    CASE
        WHEN LOWER(TRIM(a.activity)) = 'devzone downloads' THEN
            CASE
                WHEN LOWER(COALESCE(a.filepath, '')) LIKE '%.exe' 
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%installer%'
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%toolkit%'
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%.deb'
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%.rpm'
                THEN 'High'
                WHEN LOWER(COALESCE(a.filepath, '')) LIKE '%.pdf'
                  OR LOWER(COALESCE(a.filepath, '')) LIKE '%docs/%'
                THEN 'Passive'
                ELSE 'Moderate'
            END
        ELSE d.effort_level
    END AS effort_level,
    d.modality
FROM activity_sample a
LEFT JOIN activity_dict_clean d
  ON LOWER(TRIM(a.activity)) = d.activity
WHERE a.dev_contact IS NOT NULL
  AND a.activity_date IS NOT NULL
""")

print("Enriched sample size (should match activity_sample):")
print(con.execute("SELECT COUNT(*) AS rows FROM activity_sample_enriched").fetchdf())

print("\nDistribution by journey signal:")
print(con.execute("""
SELECT journey_signal, COUNT(*) AS events
FROM activity_sample_enriched
GROUP BY 1
ORDER BY events DESC
""").fetchdf())

Enriched sample size (should match activity_sample):
     rows
0  200000

Distribution by journey signal:
  journey_signal  events
0          Build  124945
1       Discover   30020
2       Evaluate   27418
3          Learn   11496
4       Champion    6121


## Step 3: Activation Gate

We filter the developer population into three buckets before doing any stage or dormancy analysis. This is the critical filter that separates real users from noise.

### Definitions

**Tourist** - lifetime active days = 1. Showed up once, never returned. Most likely a free-email signup who grabbed something and left.

**Free Email User** - returned at least once, but their entire engagement is low-intent download behavior:
- 2 or fewer Build/Champion events lifetime
- Zero high-intent product usage (no Hosted API, Brev, or community contribution)
- At least one DevZone download

These users come back occasionally but only to download more free assets. They use NVIDIA as a software distribution service rather than a development platform.

**Real User** - everyone else. Has demonstrated genuine engagement beyond free downloads. This is the population we analyze for stage and dormancy.

### Why This Matters
NVIDIA's data is dominated by tourists (87% in our sample) and free-email-only users (7%). Without filtering these out, every downstream metric is dominated by noise. The original framework labeled most of this population as "Dormant" which obscured the meaningful population entirely.

In [5]:
con.execute("""
CREATE OR REPLACE TABLE dev_activation_sample AS
WITH lifetime_stats AS (
    SELECT
        developer_id,
        MIN(activity_date) AS first_activity_date,
        MAX(activity_date) AS last_activity_date,
        COUNT(*) AS lifetime_events,
        COUNT(DISTINCT activity_date) AS lifetime_active_days,
        COUNT(DISTINCT week_start) AS lifetime_active_weeks,
        DATE_DIFF('day', MIN(activity_date), MAX(activity_date)) AS lifetime_span_days,
        DATE_DIFF('day', MAX(activity_date), CURRENT_DATE) AS days_since_last_activity,
        SUM(CASE WHEN activity_clean = 'devzone downloads' THEN 1 ELSE 0 END) AS devzone_count,
        SUM(CASE WHEN journey_signal IN ('Build', 'Champion') THEN 1 ELSE 0 END) AS build_champion_events,
        SUM(CASE WHEN modality IN ('Hosted API', 'Cloud Workspace', 'Community') THEN 1 ELSE 0 END) AS high_intent_events
    FROM activity_sample_enriched
    GROUP BY 1
)
SELECT
    *,
    CASE
        WHEN lifetime_active_days = 1 THEN 'tourist'
        WHEN build_champion_events <= 2 
          AND high_intent_events = 0 
          AND devzone_count >= 1
        THEN 'free_email_user'
        ELSE 'real_user'
    END AS user_type
FROM lifetime_stats
""")

print("Activation Gate Results:")
print(con.execute("""
SELECT 
    user_type,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_activation_sample
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())

Activation Gate Results:
         user_type  developers    pct
0          tourist      128611  87.43
1  free_email_user        9958   6.77
2        real_user        8538   5.80


## Step 4: Assign Max Stage Reached for Real Users

For each real user, we identify the highest stage they reached at any point in their lifetime. The hierarchy is:

Champion > Build > Evaluate > Learn > Discover

### Threshold Logic
Stages use 1+ event thresholds because the activation gate already filtered out single-event noise. Within the real user population, any single event in a stage qualifies the user as having reached that stage.

- **Champion**: 1+ Champion event (forum contribution, bug filed, user feedback, contest)
- **Build**: 1+ Build event (Hosted API, Brev, NGC, hackathon, or installer download) OR 1+ Hosted API call OR 1+ Brev session
- **Evaluate**: 1+ Evaluate event (DevZone non-installer download or program application)
- **Learn**: 1+ Learn event (training, webinar, conference) OR 1+ Training event
- **Discover**: 1+ Discover event (registration, on-demand view, membership signup, marketing comm)

In [6]:
con.execute("""
CREATE OR REPLACE TABLE dev_lifetime_stage_sample AS
WITH lifetime_signals AS (
    SELECT
        developer_id,
        SUM(CASE WHEN journey_signal = 'Discover' THEN 1 ELSE 0 END) AS discover_count,
        SUM(CASE WHEN journey_signal = 'Learn' THEN 1 ELSE 0 END) AS learn_count,
        SUM(CASE WHEN journey_signal = 'Evaluate' THEN 1 ELSE 0 END) AS evaluate_count,
        SUM(CASE WHEN journey_signal = 'Build' THEN 1 ELSE 0 END) AS build_count,
        SUM(CASE WHEN journey_signal = 'Champion' THEN 1 ELSE 0 END) AS champion_count,
        SUM(CASE WHEN modality = 'Hosted API' THEN 1 ELSE 0 END) AS hosted_api_count,
        SUM(CASE WHEN modality = 'Cloud Workspace' THEN 1 ELSE 0 END) AS cloud_workspace_count,
        SUM(CASE WHEN modality = 'Download' THEN 1 ELSE 0 END) AS download_count,
        SUM(CASE WHEN modality = 'Training' THEN 1 ELSE 0 END) AS training_count,
        SUM(CASE WHEN effort_level = 'High' THEN 1 ELSE 0 END) AS high_effort_count
    FROM activity_sample_enriched
    GROUP BY 1
)
SELECT
    *,
    CASE
        WHEN champion_count >= 1 THEN 'Champion'
        WHEN build_count >= 1 OR hosted_api_count >= 1 OR cloud_workspace_count >= 1 THEN 'Build'
        WHEN evaluate_count >= 1 THEN 'Evaluate'
        WHEN learn_count >= 1 OR training_count >= 1 THEN 'Learn'
        WHEN discover_count >= 1 THEN 'Discover'
        ELSE 'None'
    END AS max_stage_reached
FROM lifetime_signals
""")

print("Max stage distribution (real users only):")
print(con.execute("""
SELECT 
    s.max_stage_reached,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_activation_sample a
JOIN dev_lifetime_stage_sample s USING(developer_id)
WHERE a.user_type = 'real_user'
GROUP BY 1
ORDER BY 
    CASE s.max_stage_reached
        WHEN 'Champion' THEN 1
        WHEN 'Build' THEN 2
        WHEN 'Evaluate' THEN 3
        WHEN 'Learn' THEN 4
        WHEN 'Discover' THEN 5
        ELSE 6
    END
""").fetchdf())

Max stage distribution (real users only):
  max_stage_reached  developers    pct
0          Champion         596   6.98
1             Build        7589  88.88
2          Evaluate           8   0.09
3             Learn         217   2.54
4          Discover         128   1.50


## Step 5: Dormancy Classification

For real users, we classify dormancy based on days since last activity. The thresholds were calibrated to match NVIDIA's transactional engagement pattern (developers don't engage weekly — they engage in bursts when they have a project).

### Thresholds
- **Active**: less than 180 days since last activity (engaged within ~6 months)
- **At Risk**: 180-364 days (silent for 6 months to a year)
- **Dormant**: 365+ days (silent for over a year)

### Why These Thresholds
The original framework used 84-day thresholds, but NVIDIA's data shows activity gaps of 100+ days are normal for engaged developers. They download what they need and build offline for months. A 180-day window accommodates this transactional cadence without flagging healthy users as at-risk.

In [7]:
print("Dormancy distribution (real users only):")
print(con.execute("""
SELECT 
    CASE
        WHEN days_since_last_activity < 180 THEN 'Active (under 180 days)'
        WHEN days_since_last_activity < 365 THEN 'At Risk (180-364 days)'
        ELSE 'Dormant (365+ days)'
    END AS dormancy_state,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_activation_sample
WHERE user_type = 'real_user'
GROUP BY 1
ORDER BY MIN(days_since_last_activity)
""").fetchdf())

Dormancy distribution (real users only):
            dormancy_state  developers    pct
0  Active (under 180 days)        2937  34.40
1   At Risk (180-364 days)        1160  13.59
2      Dormant (365+ days)        4441  52.01


In [8]:
con.execute("""
CREATE OR REPLACE TABLE dev_final_status_sample AS
SELECT
    a.developer_id,
    a.user_type,
    a.lifetime_events,
    a.lifetime_active_days,
    a.days_since_last_activity,
    s.max_stage_reached,
    CASE
        WHEN a.user_type = 'tourist' THEN 'Tourist'
        WHEN a.user_type = 'free_email_user' THEN 'FreeEmail'
        WHEN a.days_since_last_activity >= 365 THEN 'Dormant_' || s.max_stage_reached
        WHEN a.days_since_last_activity >= 180 THEN 'AtRisk_' || s.max_stage_reached
        ELSE 'Active_' || s.max_stage_reached
    END AS final_status
FROM dev_activation_sample a
LEFT JOIN dev_lifetime_stage_sample s USING(developer_id)
""")

print("Final Status Distribution:")
print(con.execute("""
SELECT 
    final_status,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_final_status_sample
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())

Final Status Distribution:
        final_status  developers    pct
0            Tourist      128611  87.43
1          FreeEmail        9958   6.77
2      Dormant_Build        3623   2.46
3       Active_Build        2882   1.96
4       AtRisk_Build        1084   0.74
5   Dormant_Champion         484   0.33
6      Dormant_Learn         203   0.14
7   Dormant_Discover         125   0.08
8    AtRisk_Champion          63   0.04
9    Active_Champion          49   0.03
10      AtRisk_Learn          10   0.01
11  Dormant_Evaluate           6   0.00
12      Active_Learn           4   0.00
13   AtRisk_Discover           2   0.00
14   Active_Discover           1   0.00
15   Active_Evaluate           1   0.00
16   AtRisk_Evaluate           1   0.00


## Step 7: Channel Quality Analysis

We compare program_application_source channels by the rate at which they produce real users vs tourists. This identifies which acquisition channels deliver engaged developers and which produce primarily one-touch signups.

In [9]:
print("Channel quality breakdown:")
print(con.execute("""
SELECT 
    c.program_application_source AS channel,
    COUNT(*) AS total_signups,
    SUM(CASE WHEN a.user_type = 'tourist' THEN 1 ELSE 0 END) AS tourists,
    SUM(CASE WHEN a.user_type = 'free_email_user' THEN 1 ELSE 0 END) AS free_email,
    SUM(CASE WHEN a.user_type = 'real_user' THEN 1 ELSE 0 END) AS real_users,
    ROUND(100.0 * SUM(CASE WHEN a.user_type = 'real_user' THEN 1 ELSE 0 END) / COUNT(*), 2) AS real_user_rate_pct
FROM dev_activation_sample a
JOIN contact_sample c ON a.developer_id = c.developer_id
WHERE c.program_application_source IS NOT NULL
GROUP BY 1
HAVING COUNT(*) >= 50
ORDER BY total_signups DESC
""").fetchdf())


Channel quality breakdown:
            channel  total_signups  tourists  free_email  real_users  \
0              null          56535   48784.0      4273.0      3478.0   
1           devzone          42115   38528.0      2252.0      1335.0   
2           unknown          22798   18206.0      2983.0      1609.0   
3       api_catalog           9478    7998.0         3.0      1477.0   
4               dli           8263    8031.0        96.0       136.0   
5               gtc           4726    4132.0       259.0       335.0   
6     gtc fall 2022           1377    1273.0        47.0        57.0   
7               nod            915     787.0        36.0        92.0   
8   gtc spring 2022            563     553.0         5.0         5.0   
9              brev            144     138.0         0.0         6.0   
10              ngc             82      79.0         2.0         1.0   
11        inception             55      51.0         1.0         3.0   

    real_user_rate_pct  
0          

## Step 8: Export Results

Export key tables to CSV for further analysis, slide preparation, and team sharing.

In [10]:
os.makedirs("Data", exist_ok=True)

con.execute("COPY dev_final_status_sample TO 'Data/dev_final_status_sample.csv' (HEADER, DELIMITER ',')")
con.execute("COPY dev_activation_sample TO 'Data/dev_activation_sample.csv' (HEADER, DELIMITER ',')")
con.execute("COPY dev_lifetime_stage_sample TO 'Data/dev_lifetime_stage_sample.csv' (HEADER, DELIMITER ',')")
con.execute("COPY activity_dict_clean TO 'Data/activity_dict_clean.csv' (HEADER, DELIMITER ',')")

print("Exports complete. Files saved to Data/ folder:")
print("- dev_final_status_sample.csv (one row per developer with full classification)")
print("- dev_activation_sample.csv (lifetime stats and user_type)")
print("- dev_lifetime_stage_sample.csv (stage signal counts)")
print("- activity_dict_clean.csv (the 22-row classification dictionary)")

Exports complete. Files saved to Data/ folder:
- dev_final_status_sample.csv (one row per developer with full classification)
- dev_activation_sample.csv (lifetime stats and user_type)
- dev_lifetime_stage_sample.csv (stage signal counts)
- activity_dict_clean.csv (the 22-row classification dictionary)


In [11]:
con.close()
print("Pipeline complete.")

Pipeline complete.
